# Multi-Target Tutorial

This notebook builds a multi-target passive-sonar scenario step by step.

The current implementation in this notebook uses:
- Platform: Maneuvering `TowedArrayPlatform` with a passive towed linear array
- Propagation: `rtrsAcousticPropagationModel` with linear SSP and flat bathymetry
- Signals: `BroadbandShipSignal` plus `ColouredNoise`
- Beamforming/Detection: Frequency-domain delay-and-sum, CA-CFAR, and peak-picking
- Tracking: Bearing-only Kalman filter with JPDA and bearing wrapping

We first define global timing and reproducibility settings.

This cell seeds NumPy (`seed = 2000`), sets the simulation duration
(`sim_duration = 900 s`), sets the integration interval (`time_interval = 5 s`), and
builds `timesteps` for all downstream components.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from datetime import datetime, timedelta

import numpy as np

# Random seed for reproducibility
seed = 2000
np.random.seed(seed)

# Simulation parameters
sim_duration = timedelta(seconds=900)
time_interval = timedelta(seconds=5)

num_steps = int(sim_duration.total_seconds() / time_interval.total_seconds())
start_time = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
timesteps = [start_time + i * time_interval for i in range(num_steps)]

## Platform Setup and Generation

This section constructs a `TowedArrayPlatform` and simulates its motion over all
timesteps.

The host starts from `platform_start_vector` and follows a three-stage deterministic
trajectory: straight leg, a `-45 deg` constant-rate turn (`KnownTurnRate` at `1 deg/s`),
then another straight leg. Process noise is set to zero in these transition models.

Array configuration used by the code:
- `num_sensors = 200`
- `cable_length_m = 100.0`
- `sensor_spacing_m = 0.5`
- `array_depth_m = -50.0`

Finally, we propagate the platform by calling `platform.move(timestamp)` for each
timestamp in the simulation.

In [ ]:
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
    KnownTurnRate,
)
from stonesoup.types.groundtruth import GroundTruthState

from nereus.platform import TowedArrayPlatform

# Define the platform's initial state and transition model
platform_start_vector = np.array([-7500.0, 1.15, -2000.0, 0.25, -5.0, 0.0])
platform_position_mapping = [0, 2, 4]
platform_velocity_mapping = [1, 3, 5]
platform_turn_rate_radps = np.deg2rad(1.0)

leg1_duration_s = timedelta(seconds=405)
turn1_angle_rad = np.deg2rad(-45)
turn1_duration_s = timedelta(
    seconds=round((abs(turn1_angle_rad) / platform_turn_rate_radps) / 5.0) * 5.0
)
leg2_duration_s = sim_duration - leg1_duration_s - turn1_duration_s

straight_model = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(0.0), ConstantVelocity(0.0), ConstantVelocity(0.0)]
)

turn_rate_rad1 = np.sign(turn1_angle_rad) * platform_turn_rate_radps
planar_turn1 = KnownTurnRate(
    turn_rate=turn_rate_rad1,
    turn_noise_diff_coeffs=np.array([0.0, 0.0]),
)
depth_model = ConstantVelocity(0.0)
turning_model1 = CombinedLinearGaussianTransitionModel([planar_turn1, depth_model])

transition_models = [straight_model, turning_model1, straight_model]
transition_times = [leg1_duration_s, turn1_duration_s, leg2_duration_s]

# Define the towed array parameters
num_sensors = 200
tow_cable_length_m = 100.0
sensor_spacing_m = 0.5
array_depth_m = -50.0

# Create the towed array platform and simulate its movement over time
platform_initial_state = GroundTruthState(platform_start_vector, timestamp=start_time)
platform = TowedArrayPlatform(
    states=platform_initial_state,
    position_mapping=platform_position_mapping,
    velocity_mapping=platform_velocity_mapping,
    transition_models=transition_models,
    transition_times=transition_times,
    num_sensors=num_sensors,
    cable_length_m=tow_cable_length_m,
    sensor_spacing_m=sensor_spacing_m,
    array_depth_m=array_depth_m,
)

for timestamp in timesteps[1:]:
    platform.move(timestamp)

## Ground Truth Setup and Generation

Here we build three target truth trajectories and attach signal metadata to each
`GroundTruthState`.

Each target uses a deterministic constant-velocity transition model in 3D (`x, vx, y,
vy, z, vz`). Metadata fields include tonal amplitudes/frequencies/phases, tonal
bandwidth, and stochastic-noise parameters. We then visualize platform and target paths
with `plot_world`.

In [ ]:
import numpy as np
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

from nereus.plotter import plot_world

# Define the target's initial state and transition model
target1_start_vector = np.array([-1.5e4, 9.0, 1.2e4, -10, -5.0, 0.0])
target2_start_vector = np.array([-1.1e4, -8.0, -5.7e3, 3.6, -5.0, 0.0])
target3_start_vector = np.array([-3.0e3, 10.4, -9.7e3, 9.7, -5.0, 0.0])

target_transition_model = CombinedLinearGaussianTransitionModel([
    ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)
])
target_position_mapping = [0, 2, 4]
target_velocity_mapping = [1, 3, 5]

target_truths = []

target_tonal_bandwidth_hz = np.random.uniform(0.5, 2.0)
target_noise_amplitude_upa = 10 ** (90 / 20)
target_noise_spectral_exponent = -1.0

for sv in [target1_start_vector, target2_start_vector, target3_start_vector]:
    metadata = {
        "position_mapping": target_position_mapping,
        "velocity_mapping": target_velocity_mapping,
        "amplitudes_upa": 10 ** (np.random.uniform(87, 102, 4) / 20),
        "frequencies_hz": np.random.uniform(25.0, 200.0, 4),
        "phases_rad": np.random.uniform(0, 2 * np.pi, 4),
        "tonal_bandwidth_hz": np.random.uniform(0.5, 2.0),
        "noise_amplitude_upa": 10 ** (np.random.uniform(65, 85) / 20),
        "target_tonal_bandwidth_hz": target_tonal_bandwidth_hz,
        "target_noise_amplitude_upa": target_noise_amplitude_upa,
        "noise_spectral_exponent": target_noise_spectral_exponent,
    }
    target_states = [GroundTruthState(sv, timestamp=start_time, metadata=metadata)]
    for timestamp in timesteps[1:]:
        dt = timestamp - target_states[-1].timestamp
        new_state_vector = target_transition_model.function(
            target_states[-1], noise=False, time_interval=dt
        )
        target_states.append(
            GroundTruthState(new_state_vector, timestamp=timestamp, metadata=metadata)
        )

    target_truths.append(GroundTruthPath(target_states))

plot_world(truths=target_truths, platform=platform).show()

## Propagation Model

This section configures the acoustic environment for propagation.

The code uses a linear sound-speed profile
(`Linear(surface_speed=1500.0, gradient=0.2)`) and flat bathymetry
(`FlatBathymetry(depth=-150.0)`). Propagation is handled by
`rtrsAcousticPropagationModel` with `step_m=20.0`,
`azimuth_search_width=2.0`, `azimuth_resolution=0.5`,
`elevation_range=(-25.0, 25.0)`, and `elevation_resolution=1.0`.


In [ ]:
from nereus.models.environment import FlatBathymetry, Linear
from nereus.models.propagation import rtrsAcousticPropagationModel

ssp = Linear(surface_speed=1500.0, gradient=0.2)
bathymetry = FlatBathymetry(depth=-150.0)
attenuation_factor = 0.5

propagation_model = rtrsAcousticPropagationModel(
    ssp=ssp,
    bathymetry=bathymetry,
    use_all_frequencies=False,
    step_m=20.0,
    azimuth_search_width=2.0,
    azimuth_resolution=0.5,
    elevation_range=(-25.0, 25.0),
    elevation_resolution=1.0,
)

## Signal Model

This section defines ambient noise and source-signal models with shared STFT settings.

`sampling_rate_hz`, `frame_len`, and `hop_factor` define the signal-processing cadence
and frequency support for beamforming.

Ambient background is modeled with `ColouredNoise` for one integration interval
(`duration_s = time_interval.total_seconds()`), so fresh ambient noise is produced each
step.

Target acoustic content is generated with `BroadbandShipSignal` over the full simulation
duration. In this notebook a single source model is passed into the simulator, with
constant tonal/noise settings (`tonal_noise_is_constant=True`,
`noise_is_constant=True`).

`noise_freq_range_hz=(0, sampling_rate_hz/2)` keeps source noise within the physically
valid one-sided frequency range.


In [ ]:
from nereus.signal.ambient import ColouredNoise
from nereus.signal.anthropogenic import BroadbandShipSignal

sampling_rate_hz = 500.0
frame_len = 500
hop_factor = 2
duration_s = num_steps * time_interval.total_seconds()

ambient_amplitude_upa = 10 ** (45 / 20)
ambient_spectral_exponent = -1
ambient_noise_model = ColouredNoise(
    amplitude_upa=ambient_amplitude_upa,
    spectral_exponent=ambient_spectral_exponent,
    duration_s=time_interval.total_seconds(),
    sampling_rate_hz=sampling_rate_hz,
)

signal_model = BroadbandShipSignal(
    duration_s=duration_s,
    sampling_rate_hz=sampling_rate_hz,
    frame_len=frame_len,
    hop_factor=hop_factor,
    tonal_bandwidth_hz=target_tonal_bandwidth_hz,
    noise_amplitude_upa=target_noise_amplitude_upa,
    noise_spectral_exponent=target_noise_spectral_exponent,
    noise_freq_range_hz=(0.0, sampling_rate_hz / 2),
    tonal_noise_is_constant=True,
    noise_is_constant=True,
)

## Beamformer

This stage produces the bearing-time record (BTR) and detections.

The code uses `DelayAndSumBeamformer` (frequency domain) and steering vectors over a
full `[-pi, pi]` azimuth grid (`361` steering angles).

`BroadbandPassiveSonarArraySimulator` then combines platform geometry, propagation,
source/noise models, and steering to generate beamformed data through time.

Detections are extracted with CA-CFAR (`num_guard_cells=6`,
`num_training_cells=10`, `threshold_factor=1.05`, `mode="wrap"`) and refined with
`PeakDetector(distance=8)`. We then plot BTR outputs with and without detections.


In [ ]:
import numpy as np

from nereus.detector import CFARDetector, PassiveSonarDetector, PeakDetector
from nereus.plotter import plot_btr
from nereus.sigproc import (
    DelayAndSumBeamformer,
    SteeringCalculator,
)
from nereus.simulator import BroadbandPassiveSonarArraySimulator

steering_azimuths_rad = np.linspace(-np.pi, np.pi, 361)

beamformer = DelayAndSumBeamformer(
    sampling_rate_hz=sampling_rate_hz,
    shading=None,
    domain="frequency",
)

steering_calculator = SteeringCalculator(
    ssp=ssp,
    steering_azimuths_rad=steering_azimuths_rad,
)

fade_in_ms = 1000.0

simulator = BroadbandPassiveSonarArraySimulator(
    platform=platform,
    propagation_model=propagation_model,
    signal_models=[signal_model],
    noise_model=ambient_noise_model,
    beamformer=beamformer,
    steering_calculator=steering_calculator,
    ground_truth_paths=target_truths,
    fade_in_ms=fade_in_ms,
)

cfar_detector = CFARDetector(
    num_guard_cells=6,
    num_training_cells=10,
    threshold_factor=1.05,
    mode="wrap"
)
peak_detector = PeakDetector(distance=8)

detection_chain = [cfar_detector, peak_detector]

detector = PassiveSonarDetector(
    detection_chain=detection_chain,
    sensor_data_gen=simulator.sensor_data_gen(),
    steering_azimuths_rad=steering_azimuths_rad,
)

all_detections = list(detector.detections_gen(progress_bar=True))
snr_map = detector.snr_history

detections_for_plotter = [d for _, detections in all_detections for d in detections]
print(f"Detections: {len(detections_for_plotter)}")

In [ ]:
plot_btr(
    data=snr_map,
    timesteps=timesteps,
    steering_azimuths=np.rad2deg(steering_azimuths_rad),
).show()

plot_btr(
    timesteps=timesteps,
    steering_azimuths=np.rad2deg(steering_azimuths_rad),
    data=snr_map,
    detections=detections_for_plotter,
).show()

## Tracker

Tracking is performed in two steps.

First, Cartesian target truths are converted to relative-bearing truths using the array
reference position at each timestamp.

Second, a bearing-only Kalman/JPDA tracker is run with:
- State model: `ConstantVelocity(1e-6)` on `[bearing, bearing_rate]`
- Measurement model: `LinearGaussian` with `noise_covar = np.deg2rad(6)^2`
- Association: `PDAHypothesiser` + `JPDA`, with `prob_detect=0.85`
- Clutter model: `expected_false_alarms_per_scan = 3` over full `360 deg` field of view
- Track management: `missed_distance=6`, `min_points=60`, and `CovarianceBasedDeleter(covar_trace_thresh=0.2)`

After each update, track bearings are wrapped with `mod_bearing` before plotting truths,
detections, and tracks on the BTR.


In [ ]:
relative_bearing_truths = []

for target_truth in target_truths:
    bearing_states = []
    for state in target_truth:
        platform_state = platform.get_platform_state_at(state.timestamp)
        ref_sensor_position = np.mean(platform_state.array.state_vector, axis=1)

        target_xy = np.array([state.state_vector[0], state.state_vector[2]])
        relative_position = target_xy - ref_sensor_position[:2]
        bearing = np.arctan2(relative_position[1], relative_position[0])

        bearing_states.append(
            GroundTruthState(np.array([bearing]), timestamp=state.timestamp)
        )

    relative_bearing_truths.append(GroundTruthPath(bearing_states))

In [ ]:
import numpy as np
from stonesoup.dataassociator.neighbour import GNNWith2DAssignment
from stonesoup.dataassociator.probability import JPDA
from stonesoup.deleter.error import CovarianceBasedDeleter
from stonesoup.functions import mod_bearing
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.measures import Mahalanobis
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.models.transition.linear import ConstantVelocity
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.tracker.simple import MultiTargetMixtureTracker
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater

from nereus.plotter import plot_btr

transition_model = ConstantVelocity(0.000001)

predictor = KalmanPredictor(transition_model)

measurement_model = LinearGaussian(
    ndim_state=2,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(6) ** 2]]),
)

updater = KalmanUpdater(measurement_model=measurement_model)

fov_rad = np.deg2rad(360)
expected_false_alarms_per_scan = 3
clutter_spatial_density = expected_false_alarms_per_scan / fov_rad

hypothesiser = PDAHypothesiser(
    predictor=predictor,
    updater=updater,
    clutter_spatial_density=clutter_spatial_density,
    prob_detect=0.85,
)
data_associator = JPDA(hypothesiser=hypothesiser)

init_hypothesiser = DistanceHypothesiser(
    predictor=predictor,
    updater=updater,
    measure=Mahalanobis(),
    missed_distance=6,
)
init_associator = GNNWith2DAssignment(init_hypothesiser)

deleter = CovarianceBasedDeleter(covar_trace_thresh=0.2)

# Get bearing from first detection for prior state
first_detection = next(
    det
    for _, det_set in all_detections
    if det_set
    for det in det_set
)

initial_bearing = float(first_detection.state_vector[0, 0])

prior_state = GaussianState(
    np.array([[first_detection], [0.0]]),
    np.diag([np.deg2rad(5) ** 2, np.deg2rad(0.5) ** 2]),
    timestamp=start_time,
)

initiator = MultiMeasurementInitiator(
    prior_state=prior_state,
    measurement_model=measurement_model,
    deleter=deleter,
    data_associator=init_associator,
    updater=updater,
    min_points=60,
)

kf = MultiTargetMixtureTracker(
    initiator=initiator,
    deleter=deleter,
    detector=all_detections,
    data_associator=data_associator,
    updater=updater,
)

tracks = set()

for _, current_tracks in kf:
    for track in current_tracks:
        track[-1].state_vector[0, 0] = mod_bearing(float(track[-1].state_vector[0, 0]))
    tracks |= current_tracks

plot_btr(
    timesteps=timesteps,
    steering_azimuths=np.rad2deg(steering_azimuths_rad),
    truths=relative_bearing_truths,
    detections=detections_for_plotter,
    tracks=tracks,
).show()